# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [8]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [9]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0381


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_c7852de926cdeba9,2026-11-10 06:03:04.911000000,u_2e1d30c7,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_c99d34', 'ts': '2026-11-10 ...","[p_8fbf52, p_5969fa, p_0c3c0c, p_40ec57, p_8d8...","[0.013001, 0.303182, 0.215678, 0.293852, 0.040...",NaN,NaN
1,r_be003a7375e44b6d,2026-11-13 02:04:20.325000000,u_c162ac54,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_73a847', 'ts': '2026-11-13 ...","[p_c88f21, p_3c342d, p_59964d, p_1a7fd6, p_1bb...","[0.015102, 0.082563, 0.011842, 0.026765, 0.001...",NaN,NaN
2,r_e4e8a85dfd159ad5,2026-11-11 01:56:10.590000000,u_7bef0d88,"{'device': 'web', 'page': 'product', 'region':...","[{'product_id': 'p_1d1d3f', 'ts': '2026-11-11 ...","[p_8ca8ed, p_4077a5, p_7a1c91, p_92d45d, p_b82...","[0.218886, 0.009675, 0.008384, 0.038402, 0.020...",NaN,NaN


In [11]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [12]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [13]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [ ]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

___

Métricas:
- hit_rate@10 = (# clicked products in top 10) / (# observed clicks)
- IPC_ctr_slot1
- SNIPS_ctr_slot1
- slot1_matches: # observations where the new policy's first recommendation matches the logging policy's first recommendation

In [15]:
DIMENSIONS = 128

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ItemTower(nn.Module):
    def __init__(
        self,
        n_categories,
        n_subcategories,
        n_brands,
        dim=128,
    ):
        super().__init__()

        self.category_emb = nn.Embedding(n_categories, 16)
        self.subcategory_emb = nn.Embedding(n_subcategories, 16)
        self.brand_emb = nn.Embedding(n_brands, 16)

        self.price_net = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
        )

        self.mlp = nn.Sequential(
            nn.Linear(64 + 16 + 16 + 16 + 8, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, dim),
        )

    def forward(self, product, category, subcategory, brand, price):
        x = torch.cat([
            self.product_emb(product),
            self.category_emb(category),
            self.subcategory_emb(subcategory),
            self.brand_emb(brand),
            self.price_net(price.unsqueeze(-1)),
        ], dim=-1)

        return F.normalize(self.mlp(x), dim=-1)


In [ ]:
class UserTower(nn.Module):
    def __init__(
        self,
        n_devices,
        n_regions,
        n_pages,
        item_dim=128,
        dim=128,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=item_dim,
            hidden_size=128,
            batch_first=True,
        )

        self.device_emb = nn.Embedding(n_devices, 8)
        self.region_emb = nn.Embedding(n_regions, 16)
        self.page_emb = nn.Embedding(n_pages, 8)

        self.mlp = nn.Sequential(
            nn.Linear(128 + 8 + 16 + 8, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, dim),
        )

    def forward(self, history_embeddings, device, region, page):
        # history_embeddings: [batch, sequence, item_dim]

        _, hidden = self.gru(history_embeddings)
        history_vector = hidden[-1]

        context = torch.cat([
            self.device_emb(device),
            self.region_emb(region),
            self.page_emb(page),
        ], dim=-1)

        x = torch.cat([history_vector, context], dim=-1)

        return F.normalize(self.mlp(x), dim=-1)


In [18]:

def recommendation_loss(
    user_vectors,
    positive_vectors,
    negative_vectors,
    weights=None,
):
    """
    user_vectors:     [B, D]
    positive_vectors: [B, D]
    negative_vectors: [B, K, D]
    """

    positive_scores = (
        user_vectors * positive_vectors
    ).sum(dim=-1, keepdim=True)

    negative_scores = torch.einsum(
        "bd,bkd->bk",
        user_vectors,
        negative_vectors,
    )

    logits = torch.cat(
        [positive_scores, negative_scores],
        dim=1,
    )

    # Class 0 is always the positive product.
    targets = torch.zeros(
        len(user_vectors),
        dtype=torch.long,
        device=user_vectors.device,
    )

    loss = F.cross_entropy(
        logits,
        targets,
        reduction="none",
    )

    if weights is not None:
        loss = loss * weights

    return loss.mean()


In [ ]:
item_tower = ItemTower(
    n_products=n_products,
    n_categories=n_categories,
    n_subcategories=n_subcategories,
    n_brands=n_brands,
    dim=128,
)

user_tower = UserTower(
    n_devices=n_devices,
    n_regions=n_regions,
    n_pages=n_pages,
    item_dim=128,
    dim=128,
)

In [ ]:
# Encode the user's history.
history_embeddings = item_tower(
    history_product,
    history_category,
    history_subcategory,
    history_brand,
    history_price,
)

# Encode positive and negative candidates using that same tower.
positive_embeddings = item_tower(...)
negative_embeddings = item_tower(...)

# Generate the user representation.
user_vector = user_tower(
    history_embeddings,
    device,
    region,
    page,
)

In [19]:
model_params = list(item_tower.parameters()) + \
               list(user_tower.parameters())

optimizer = torch.optim.AdamW(
    model_params,
    lr=1e-3,
    weight_decay=1e-5,
)

for batch in train_loader:
    optimizer.zero_grad()

    # Encode history products with item_tower.
    # Encode positive and negative candidates.
    # Run user_tower and compute recommendation_loss.

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_params, 1.0)
    optimizer.step()

NameError: name 'item_tower' is not defined

In [ ]:
@torch.no_grad()
def encode_catalog(item_tower, catalog, device="cpu"):
    item_tower.eval()

    tensors = {
        k: torch.as_tensor(
            catalog[k].to_numpy(),
            dtype=torch.long if k != "price" else torch.float32,
            device=device,
        )
        for k in [
            "product", "category", "subcategory",
            "brand", "price"
        ]
    }

    vectors = item_tower(
        tensors["product"],
        tensors["category"],
        tensors["subcategory"],
        tensors["brand"],
        tensors["price"],
    )

    return vectors.cpu()

In [ ]:
item_vectors = encode_catalog(item_tower, catalog)

torch.save({
    "item_vectors": item_vectors,
    "product_ids": catalog.product_id.tolist(),
}, "item_vectors.pt")

In [ ]:
@torch.no_grad()
def recommend(
    request,
    user_tower,
    item_vectors,
    product_ids,
    item_index,
    active_mask,
    context_indices,
    device="cpu",
):
    user_tower.eval()

    history_ids = [
        x["product_id"]
        for x in request["recent_views"][-30:]
    ]

    history_indices = [
        item_index[p]
        for p in history_ids
        if p in item_index
    ]

    if history_indices:
        history = item_vectors[
            history_indices
        ].unsqueeze(0).to(device)
    else:
        history = torch.zeros(
            1, 1, item_vectors.shape[-1],
            device=device,
        )

    user_vector = user_tower(
        history,
        torch.tensor([context_indices["device"]], device=device),
        torch.tensor([context_indices["region"]], device=device),
        torch.tensor([context_indices["page"]], device=device),
    )

    scores = (
        user_vector @ item_vectors.to(device).T
    ).squeeze(0)

    scores[~active_mask] = -float("inf")

    # Optional: exclude recent views.
    for p in history_ids:
        if p in item_index:
            scores[item_index[p]] = -float("inf")

    top_indices = torch.topk(scores, k=10).indices.tolist()

    return [product_ids[i] for i in top_indices]